## 1-load and setup

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from src.preprocess import clean

imdb = pd.read_csv("data/raw/imdb_50k.csv")
imdb["label"] = (imdb["sentiment"] == "positive").astype(int)
pl = pd.read_csv("data/raw/pang_lee_2000.csv")
own = pd.read_csv("data/own/own_reviews.csv")
print(imdb.shape, pl.shape, own.shape)

(50000, 3) (2000, 2) (300, 10)


## 2-deduplicate and split

In [2]:
n0 = len(imdb)
imdb = imdb.drop_duplicates(subset="review").reset_index(drop=True)
print(n0 - len(imdb), len(imdb), imdb["label"].value_counts().to_dict())

train, rest = train_test_split(imdb, test_size=0.2, stratify=imdb["label"], random_state=42)
val, test = train_test_split(rest, test_size=0.5, stratify=rest["label"], random_state=42)
print(len(train), len(val), len(test))
print(train["label"].mean().round(3), val["label"].mean().round(3), test["label"].mean().round(3))

418 49582 {1: 24884, 0: 24698}
39665 4958 4959
0.502 0.502 0.502


## 3-test the cleaning function

In [3]:
tests = ["This isn't a good movie, I can't recommend it.",
         "Not bad at all",
         "<br />Loved it!!! 10/10, wouldn't change a thing."]
for s in tests:
    print(clean(s), "|", clean(s, stop="all"), "|", clean(s, stop="keep_neg"))

this is not good movie can not recommend it | good movie recommend | not good movie not recommend
not bad at all | bad | not bad
loved it would not change thing | loved would change thing | loved would not change thing


## 4-clean everything

In [4]:
def add_clean(d):
    d = d.copy()
    d["clean"] = d["review"].apply(clean)
    d["clean_nostop"] = d["review"].apply(lambda x: clean(x, stop="keep_neg"))
    return d

train, val, test, pl, own = [add_clean(d) for d in (train, val, test, pl, own)]
print(train[["review", "clean"]].iloc[0].to_dict())

{'review': 'I am one of Jehovah\'s Witnesses and I also work in an acute care medical facility. Over the years I have seen people die from hemolytic reactions to blood transfusions, have attended numerous conferences on blood born pathogens, and have seen several patients become seriously ill from pathogens induced by transfused blood. I have also heard several Jehovah\'s Witnesses being told that they will die if they refuse blood and after 26 years in the field I have never actually seen it happen, leaving the question, "is it really unreasonable to refuse blood transfusions or is the community at large benefiting from the battle on this issue?" The issue for Jehovah\'s Witnesses is a moral one. "You must abstain from blood" is not an ambiguous statement. Thank you for this movie and allowing comments on it.', 'clean': 'am one of jehovah witness and also work in an acute care medical facility over the year have seen people die from hemolytic reaction to blood transfusion have attende

## 5-integrity and leakage checks

In [5]:
a, b, c = set(train["review"]), set(val["review"]), set(test["review"])
print("train/val/test overlap:", len(a & b), len(a & c), len(b & c))

imdb_all = set(imdb["review"].str.strip())
print("own in IMDB:", own["review"].str.strip().isin(imdb_all).sum(),
      "| Pang&Lee in IMDB:", pl["review"].str.strip().isin(imdb_all).sum())

for name, d in (("train", train), ("val", val), ("test", test), ("pang_lee", pl), ("own", own)):
    print(name, len(d), d["clean"].str.len().eq(0).sum(), round(d["clean"].str.split().str.len().mean(), 1))

raw_vocab = set(" ".join(train["review"].str.lower()).split())
clean_vocab = set(" ".join(train["clean"]).split())
print(len(raw_vocab), len(clean_vocab))

train/val/test overlap: 0 0 0
own in IMDB: 0 | Pang&Lee in IMDB: 0
train 39665 0 219.2
val 4958 0 217.9
test 4959 0 218.8
pang_lee 2000 0 624.6
own 300 0 121.6
340479 81882


## 6-compare 5 cleaning variants 

In [6]:
variants = {
    "A raw lowercase": lambda s: s.lower(),
    "B html and punctuation": lambda s: clean(s, stop="none", lemmatize=False),
    "C + lemmatization": lambda s: clean(s, stop="none", lemmatize=True),
    "D + stop words removed": lambda s: clean(s, stop="all", lemmatize=True),
    "E + stop words, negations kept": lambda s: clean(s, stop="keep_neg", lemmatize=True),
}

rows = []
for name, f in variants.items():
    xtr = train["review"].apply(f)
    xva = val["review"].apply(f)
    for ng in ((1, 1), (1, 2)):
        vec = TfidfVectorizer(ngram_range=ng, min_df=3, sublinear_tf=True)
        Xtr = vec.fit_transform(xtr)
        Xva = vec.transform(xva)
        m = LogisticRegression(max_iter=2000, C=4).fit(Xtr, train["label"])
        p = m.predict(Xva)
        rows.append((name, ng, Xtr.shape[1],
                     round(accuracy_score(val["label"], p), 4),
                     round(f1_score(val["label"], p, average="macro"), 4)))
        print(rows[-1], flush=True)

res = pd.DataFrame(rows, columns=["variant", "ngram", "n_features", "val_acc", "val_macro_f1"])
res

('A raw lowercase', (1, 1), 43613, 0.8985, 0.8985)
('A raw lowercase', (1, 2), 379707, 0.9141, 0.9141)
('B html and punctuation', (1, 1), 43089, 0.9004, 0.9003)
('B html and punctuation', (1, 2), 373042, 0.9104, 0.9104)
('C + lemmatization', (1, 1), 38071, 0.8981, 0.8981)
('C + lemmatization', (1, 2), 357647, 0.9155, 0.9155)
('D + stop words removed', (1, 1), 37964, 0.8969, 0.8969)
('D + stop words removed', (1, 2), 277659, 0.905, 0.905)
('E + stop words, negations kept', (1, 1), 37976, 0.8992, 0.8991)
('E + stop words, negations kept', (1, 2), 280182, 0.9111, 0.911)


,variant,ngram,n_features,val_acc,val_macro_f1
0,A raw lowercase,"(1, 1)",43613,0.8985,0.8985
1,A raw lowercase,"(1, 2)",379707,0.9141,0.9141
2,B html and punctuation,"(1, 1)",43089,0.9004,0.9003
3,B html and punctuation,"(1, 2)",373042,0.9104,0.9104
4,C + lemmatization,"(1, 1)",38071,0.8981,0.8981
5,C + lemmatization,"(1, 2)",357647,0.9155,0.9155
6,D + stop words removed,"(1, 1)",37964,0.8969,0.8969
7,D + stop words removed,"(1, 2)",277659,0.9050,0.9050
8,"E + stop words, negations kept","(1, 1)",37976,0.8992,0.8991
9,"E + stop words, negations kept","(1, 2)",280182,0.9111,0.9110


## 7- save

In [7]:
cols = ["review", "clean", "clean_nostop", "label"]
for name, d in (("train", train), ("val", val), ("test", test), ("external_pang_lee", pl)):
    d[cols].to_csv(f"data/processed/{name}.csv", index=False)
own[cols + ["film", "genre", "hard_case"]].to_csv("data/processed/own_reviews_processed.csv", index=False)

## 8-reload and verify 

In [8]:
for name in ("train", "val", "test", "external_pang_lee", "own_reviews_processed"):
    d = pd.read_csv(f"data/processed/{name}.csv")
    print(name, d.shape, d["clean"].isnull().sum(), list(d.columns))

train (39665, 4) 0 ['review', 'clean', 'clean_nostop', 'label']
val (4958, 4) 0 ['review', 'clean', 'clean_nostop', 'label']
test (4959, 4) 0 ['review', 'clean', 'clean_nostop', 'label']
external_pang_lee (2000, 4) 0 ['review', 'clean', 'clean_nostop', 'label']
own_reviews_processed (300, 7) 0 ['review', 'clean', 'clean_nostop', 'label', 'film', 'genre', 'hard_case']
